# A9 Prioritization & Privacy-Safe Export (CPU)

Menjalankan prioritization destinasi dan export privacy-safe (untuk app) melalui
`sipature_ml.a9.run_prioritization` dan `run_export`, lalu menyiapkan expert
review queue dan sensitivity analysis. Ikuti `docs/a9-inference-priority-report.md`.

Input: hasil notebook `08` (`a9/<run>-aggregate/`) dan
`data/processed/canonical_destinations.parquet` (notebook `02`).
Output: `a9/<run>-prioritize/` dan `a9/<run>-export/` di Drive.

`app-export.json` adalah output aman (tanpa teks review) untuk integrasi app.


## Step 1 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Konfigurasi path & parameter


In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")

# RUBAH agar sama persis dengan run ID notebook 08 (lihat output notebook 08).
RUN_ID = "20260813-1713_a9-tfidf-lexical-v1"

DESTINATIONS_PATH = DRIVE_ROOT / "data" / "processed" / "canonical_destinations.parquet"
AGGREGATION_DIR = DRIVE_ROOT / "a9" / f"{RUN_ID}-aggregate"
PRIORITIZATION_DIR = DRIVE_ROOT / "a9" / f"{RUN_ID}-prioritize"
EXPORT_DIR = DRIVE_ROOT / "a9" / f"{RUN_ID}-export"

PROJECT_DIR = Path("/content/hackathon/ml")

print("Run ID:", RUN_ID)
print("Destinations:", DESTINATIONS_PATH)
print("Aggregation dir:", AGGREGATION_DIR)
print("Prioritization dir:", PRIORITIZATION_DIR)
print("Export dir:", EXPORT_DIR)


## Step 3 — Clone repository dari GitHub


In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


## Step 4 — Verifikasi commit terbaru (git log)


In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


## Step 5 — Install dependencies (CPU profile)


In [ ]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-dev.lock.txt
!python -m pip install --no-deps -e .


**RESTART WAJIB.** Setelah install, restart runtime agar numpy/sklearn lama
tidak ter-cache di memori.

1. **Runtime > Restart session**
2. Jalankan ulang **Step 1** (mount) dan **Step 2** (config)
3. Step 3–5 **tidak perlu diulang**

Lalu lanjut ke **Step 6**.


## Step 6 — Verifikasi versi package (setelah restart)


In [ ]:
import numpy
import pandas
import pyarrow

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)


## Step 7 — Import modul sipature_ml


In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"
assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml
print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


## Step 8 — Verifikasi input (aggregation + destinations)


In [ ]:
from pathlib import Path

assert DESTINATIONS_PATH.is_file(), f"Destinations tidak ditemukan: {DESTINATIONS_PATH}"
assert AGGREGATION_DIR.is_dir(), f"Aggregation dir tidak ditemukan: {AGGREGATION_DIR}"

required = [
    AGGREGATION_DIR / "destination-aspect-signals.parquet",
    AGGREGATION_DIR / "evidence.parquet",
    AGGREGATION_DIR / "manifest.json",
]
for path in required:
    print(f"{path.name}:", "ADA" if path.is_file() else "HILANG")
    assert path.is_file()

print("\nInput A9 lengkap.")


## Step 9 — Jalankan prioritization


In [ ]:
from sipature_ml.a9 import run_prioritization

assert not PRIORITIZATION_DIR.exists(), f"Sudah ada: {PRIORITIZATION_DIR}"

summary = run_prioritization(
    aggregation_dir=AGGREGATION_DIR,
    destinations_path=DESTINATIONS_PATH,
    output_dir=PRIORITIZATION_DIR,
)

print("A9 version:", summary["a9_version"])
print("Destinations:", summary["destinations"])
print("With priority:", summary["with_priority"])
print("Severity status:", summary["severity_status"])


## Step 10 — Jalankan privacy-safe export


In [ ]:
from sipature_ml.a9 import run_export

assert not EXPORT_DIR.exists(), f"Sudah ada: {EXPORT_DIR}"

summary = run_export(
    prioritization_dir=PRIORITIZATION_DIR,
    aggregation_dir=AGGREGATION_DIR,
    output_dir=EXPORT_DIR,
)

print("A9 version:", summary["a9_version"])
print("Destinations:", summary["destinations"])
print("Output:", summary["output"])
print("Restricted:", summary["restricted"])


## Step 11 — Expert review queue + sensitivity (restricted)


In [ ]:
import json

import pandas as pd

from sipature_ml.a9 import build_expert_review_queue, weight_sensitivity
from sipature_ml.config import load_config

prioritized = pd.read_parquet(
    PRIORITIZATION_DIR / "prioritized-destinations.parquet"
)
evidence = pd.read_parquet(AGGREGATION_DIR / "evidence.parquet")

queue = build_expert_review_queue(prioritized, evidence, size=25)
queue.to_csv(PRIORITIZATION_DIR / "expert-review-queue.csv", index=False)
print("Expert review queue:", len(queue), "destinasi (restricted)")

weights = load_config("scoring")["priority_weights"]
sensitivity = weight_sensitivity(prioritized, weights)
(PRIORITIZATION_DIR / "sensitivity.json").write_text(
    json.dumps(sensitivity, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print("\nSensitivity (top-20 Jaccard):")
for scenario, result in sensitivity["scenarios"].items():
    print(f"  {scenario}: {result['top20_jaccard']}")


## Step 12 — Verifikasi output & hash artifact


In [ ]:
import json
from pathlib import Path

from sipature_ml.manifest import sha256_file

for name, directory in (("prioritize", PRIORITIZATION_DIR), ("export", EXPORT_DIR)):
    manifest = json.loads(
        (directory / "manifest.json").read_text(encoding="utf-8")
    )
    print(f"=== {name} ===")
    print("  Stage:", manifest["stage"])
    print("  A9 version:", manifest["a9_version"])
    errors = []
    for relative, expected in manifest["artifact_hashes"].items():
        path = directory / relative
        if not path.is_file():
            errors.append(f"missing: {relative}")
        elif sha256_file(path) != expected:
            errors.append(f"hash mismatch: {relative}")
    print(f"  Artifact check: {len(manifest['artifact_hashes'])} file, {len(errors)} masalah")
    assert not errors, errors

print("\nSeluruh output A9 valid terhadap manifest.")


## Step 13 — Run summary


In [ ]:
import json
from pathlib import Path

export = json.loads(
    (EXPORT_DIR / "app-export.json").read_text(encoding="utf-8")
)

print("RUN ID:", RUN_ID)
print("EXPORT DIR:", EXPORT_DIR)
print("APP EXPORT:", EXPORT_DIR / "app-export.json")

print("\nAPP EXPORT SUMMARY:")
print("  schema_version:", export["schema_version"])
print("  model_version:", export["model_version"])
print("  destinations:", len(export["destinations"]))
print("  limitations:", len(export["limitations"]))

print("\nREMINDER: app-export.json adalah output privacy-safe untuk app.")
print("Expert review queue & sensitivity tetap restricted di Drive.")
